In [1]:
import pickle
import random
import numpy as np

def evaluate_rosso_policy_worst_case_expected(
    strategy_path,
    rollouts_num=100,
    delays=(1,),
    protect_current_vertex=False,
):

    # =========================
    # LOAD STRATEGY
    # =========================

    with open(strategy_path, "rb") as f:
        P = pickle.load(f)
    P = np.array(P)
    # =========================
    # SAN FRANCISCO INSTANCE
    # =========================

    adj_matrix = [
        [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
        [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
        [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
        [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
        [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
        [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
        [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
        [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
        [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
        [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
        [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
        [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1],
    ]

    coverage = [
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    ]

    target_values = [1] * 12

    attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]

    num_nodes = len(adj_matrix)

    # =========================
    # HELPERS
    # =========================

    def sample_next(current):
        return random.choices(
            range(num_nodes),
            weights=P[current],
        )[0]

    def expected_capture_probability(path, target, tau):
        survive_prob = 1.0
        
        if protect_current_vertex:
            survive_prob *= (1.0 - coverage[path[0]][target])

        for i in range(len(path) - 1):
            pos = path[i]
            next_pos = path[i + 1]

            tau -= adj_matrix[pos][next_pos]

            if tau >= 0:
                survive_prob *= (1.0 - coverage[next_pos][target])
            else:
                break

        return 1.0 - survive_prob

    rollout_length = max(delays) + max(attack_duration) + 2

    # =========================
    # MAIN LOOP
    # =========================

    observation_rewards = {}

    for _ in range(rollouts_num):
        if _ % 100000 == 0:
            print(f"Rollout {_}/{rollouts_num}")
        rollout = []

        current = random.randrange(num_nodes)
        rollout.append(current)

        while len(rollout) < rollout_length:
            current = sample_next(current)
            rollout.append(current)
        #print("Rollout:", rollout)
        for delay in delays:

            observation = (rollout[delay - 1],)

            for target in range(num_nodes):

                tau = attack_duration[target]

                attack_path = rollout[(delay - 1):]

                capture_prob = expected_capture_probability(
                    attack_path,
                    target,
                    tau,
                )

                reward = capture_prob * target_values[target]

                key = (observation, target)

                if key not in observation_rewards:
                    observation_rewards[key] = []

                observation_rewards[key].append(reward)

    # =========================
    # WORST CASE
    # =========================

    obs_target_rewards = {}
    obs_target_counts = {}

    for key, vals in observation_rewards.items():
        obs_target_rewards[key] = sum(vals) / len(vals)
        obs_target_counts[key] = len(vals)

    worst_pair = None
    worst_value = float("inf")
    count = 0

    for pair, value in obs_target_rewards.items():
        if value < worst_value:
            worst_value = value
            worst_pair = pair
            count = obs_target_counts[pair]

    print("\nWorst case:", worst_pair)
    print("\nWorst-case value:", worst_value)
    print(f"\nWorst-case value in %: {100 * worst_value:.2f}")
    print("\nCount:", count)
    print(obs_target_rewards)
    print(obs_target_counts)
    return worst_value

In [4]:
strategy_path = "../results/local/demo_1/strategy_best_5_initializations.pkl"

evaluate_rosso_policy_worst_case_expected(
    strategy_path,
    rollouts_num=1000000,
    delays=(1,),
    protect_current_vertex=True,
)

Rollout 0/1000000
Rollout 100000/1000000
Rollout 200000/1000000
Rollout 300000/1000000
Rollout 400000/1000000
Rollout 500000/1000000
Rollout 600000/1000000
Rollout 700000/1000000
Rollout 800000/1000000
Rollout 900000/1000000

Worst case: ((6,), 8)

Worst-case value: 0.04705712184949917

Worst-case value in %: 4.71

Count: 82963
{((3,), 0): 0.15836919156710497, ((3,), 1): 0.270066588139138, ((3,), 2): 0.15091708935310946, ((3,), 3): 1.0, ((3,), 4): 0.07388398759585567, ((3,), 5): 0.1544989062237073, ((3,), 6): 0.1886704007307868, ((3,), 7): 0.12779153345032332, ((3,), 8): 0.09646866511214212, ((3,), 9): 0.123320272121926, ((3,), 10): 0.06132358950936321, ((3,), 11): 0.0744969831005553, ((2,), 0): 0.38917933713847186, ((2,), 1): 0.09819415799318912, ((2,), 2): 1.0, ((2,), 3): 0.13339968343805458, ((2,), 4): 0.062185236702000095, ((2,), 5): 0.16184229459446497, ((2,), 6): 0.13754856348026284, ((2,), 7): 0.1588205669336659, ((2,), 8): 0.0612259580795242, ((2,), 9): 0.1480166914480311, ((2,

0.04705712184949917

In [5]:
strategy_path = "../results/local/demo_1/strategy_best_5_initializations.pkl"

evaluate_rosso_policy_worst_case_expected(
    strategy_path,
    rollouts_num=1000000,
    delays=(1,2),
    protect_current_vertex=True,
)

Rollout 0/1000000
Rollout 100000/1000000
Rollout 200000/1000000
Rollout 300000/1000000
Rollout 400000/1000000
Rollout 500000/1000000
Rollout 600000/1000000
Rollout 700000/1000000
Rollout 800000/1000000
Rollout 900000/1000000

Worst case: ((6,), 8)

Worst-case value: 0.04655842353594227

Worst-case value in %: 4.66

Count: 172944
{((10,), 0): 0.07335480459549104, ((10,), 1): 0.1616868446171912, ((10,), 2): 0.12797604741074434, ((10,), 3): 0.1677986004765796, ((10,), 4): 0.16124734756662845, ((10,), 5): 0.3837564636968569, ((10,), 6): 0.16150143179898505, ((10,), 7): 0.09728679242691644, ((10,), 8): 0.1488452901710605, ((10,), 9): 0.1383935009373648, ((10,), 10): 1.0, ((10,), 11): 0.15051400553491598, ((5,), 0): 0.11454870230089212, ((5,), 1): 0.09352845627191718, ((5,), 2): 0.14172726306681038, ((5,), 3): 0.13939449142035937, ((5,), 4): 0.2622521113860509, ((5,), 5): 1.0, ((5,), 6): 0.21559667845703095, ((5,), 7): 0.09016339522009523, ((5,), 8): 0.12195689685708358, ((5,), 9): 0.1468785

0.04655842353594227

In [7]:
strategy_path = "../results/local/demo_1/strategy_best_5_initializations.pkl"

evaluate_rosso_policy_worst_case_expected(
    strategy_path,
    rollouts_num=1000000,
    delays=(1,2,3),
    protect_current_vertex=True,
)

Rollout 0/1000000
Rollout 100000/1000000
Rollout 200000/1000000
Rollout 300000/1000000
Rollout 400000/1000000
Rollout 500000/1000000
Rollout 600000/1000000
Rollout 700000/1000000
Rollout 800000/1000000
Rollout 900000/1000000

Worst case: ((6,), 8)

Worst-case value: 0.04597766035918048

Worst-case value in %: 4.60

Count: 265716
{((11,), 0): 0.08170893063315889, ((11,), 1): 0.11559173766844487, ((11,), 2): 0.09951626088202328, ((11,), 3): 0.16810777100767746, ((11,), 4): 0.10327493068372395, ((11,), 5): 0.29394736176775466, ((11,), 6): 0.15246209727032925, ((11,), 7): 0.08554344803175486, ((11,), 8): 0.14805029538425235, ((11,), 9): 0.19821505321973049, ((11,), 10): 0.43432019484404893, ((11,), 11): 1.0, ((10,), 0): 0.07376059209445684, ((10,), 1): 0.16207716165685962, ((10,), 2): 0.12707614713469956, ((10,), 3): 0.1686957109867919, ((10,), 4): 0.15999497569977875, ((10,), 5): 0.38095307110351023, ((10,), 6): 0.1616665217349295, ((10,), 7): 0.09778061199841541, ((10,), 8): 0.1487289486

0.04597766035918048